# Basic Batch Prediction Example

This notebook demonstrates:
1. Creating a batch of soil organic carbon predictions
2. Polling for batch completion
3. Retrieving the results (GeoTIFF URLs)

## Setup

First, install the required package and set up authentication:

In [ ]:
# Install the package (uncomment if needed)
# !pip install --pre spatialise

In [ ]:
import os
import time

from spatialise import SpatialiseSoilPrediction

# Set your API key (or use environment variable)
# os.environ["SPATIALISE_API_KEY"] = "your-api-key-here"

# Initialize the client
client = SpatialiseSoilPrediction(
    api_key=os.environ.get("SPATIALISE_API_KEY"),
)

print("✓ Client initialized")

## Step 1: Define Area of Interest

Define a polygon for the geographic area you want to analyze. Coordinates are in [longitude, latitude] format.

In [ ]:
# Define the area of interest (polygon coordinates in longitude, latitude)
# This example uses a small area in the Netherlands
area_of_interest = {
    "type": "Polygon",
    "coordinates": [
        [
            [6.7, 52.8],  # Bottom-left
            [6.70074, 52.8],  # Bottom-right
            [6.70074, 52.80045],  # Top-right
            [6.7, 52.80045],  # Top-left
            [6.7, 52.8],  # Close the polygon
        ]
    ],
}

print(f"Area of interest defined: {area_of_interest}")

## Step 2: Create a Batch

Submit multiple prediction jobs as a batch. Each job specifies a year and polygon.

In [ ]:
# Create a batch with prediction jobs
print("Creating batch prediction...")
batch = client.batch.create(
    jobs=[
        {
            "year": 2021,
            "polygon": area_of_interest,
        },
        {
            "year": 2022,
            "polygon": area_of_interest,
        },
    ],
    metadata={
        "project": "Soil Analysis Example",
        "location": "Netherlands",
    },
)

print(f"\n✓ Batch created successfully!")
print(f"  Batch ID: {batch.batch_id}")
print(f"  Total jobs: {batch.total_jobs}")
print(f"  Status: {batch.status}")
print(f"  Created at: {batch.created_at}")

## Step 3: Poll for Completion

Check the batch status periodically until all predictions are complete.

In [ ]:
# Poll for batch completion
print("Waiting for predictions to complete...\n")
max_retries = 60  # Maximum number of polling attempts
retry_delay = 5  # Seconds between polling attempts

for attempt in range(max_retries):
    # Retrieve the batch status
    status = client.batch.retrieve_status(batch.batch_id)

    print(
        f"  [{attempt * retry_delay}s] Status: {status.status} | "
        f"Completed: {status.completed_jobs}/{status.total_jobs} | "
        f"Failed: {status.failed_jobs} | "
        f"Pending: {status.pending_jobs}"
    )

    # Check if all jobs are completed or failed
    if status.status in ["completed", "failed"]:
        print("\n✓ Batch processing complete!")
        break

    # Wait before polling again
    time.sleep(retry_delay)
else:
    print("\n⚠ Timeout: Batch did not complete within the expected time.")

## Step 4: Retrieve Results

Get the GeoTIFF download URLs for completed predictions.

In [ ]:
# Display results
print("\nResults:")
print("=" * 80)

for idx, job in enumerate(status.jobs, 1):
    print(f"\nJob {idx}:")
    print(f"  Job ID: {job.job_id}")
    print(f"  Status: {job.status}")
    print(f"  Created: {job.created_at}")
    print(f"  Updated: {job.updated_at}")

    if job.status == "completed" and job.signed_cog_url:
        print(f"  ✓ GeoTIFF URL: {job.signed_cog_url}")
        print(f"    URL created: {job.signed_cog_url_created_at}")
        print(f"    (Valid for 6 hours from creation time)")
        print(f"\n    Download with:")
        print(f"    curl -o result_{idx}.tif '{job.signed_cog_url}'")
    elif job.status == "failed":
        print(f"  ✗ Error: {job.error_message}")

print("\n" + "=" * 80)
print(f"Summary: {status.completed_jobs} completed, {status.failed_jobs} failed")

## Optional: Download GeoTIFFs

Download the GeoTIFF files directly from the notebook.

In [ ]:
from pathlib import Path

import httpx

# Create output directory
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

print("Downloading GeoTIFF files...\n")

for idx, job in enumerate(status.jobs, 1):
    if job.status == "completed" and job.signed_cog_url:
        output_file = output_dir / f"prediction_{job.job_id}.tif"
        print(f"Downloading job {idx} to {output_file}...")

        try:
            with httpx.stream("GET", job.signed_cog_url, timeout=60.0) as response:
                response.raise_for_status()

                with open(output_file, "wb") as f:
                    for chunk in response.iter_bytes(chunk_size=8192):
                        f.write(chunk)

            print(f"  ✓ Downloaded {output_file.stat().st_size:,} bytes\n")
        except Exception as e:
            print(f"  ✗ Download failed: {e}\n")

print(f"\n✓ All files downloaded to {output_dir.absolute()}")